In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

stations_df = pd.read_csv(r"C:\Users\LIKITH REDDY\Applied data science\Project\stations_data.csv")

In [2]:
df = pd.read_csv(r"C:\Users\LIKITH REDDY\Applied data science\Project\rsu_training_45min.csv")

In [3]:
df = df.sort_values(by=['station_id', 'timestamp'])

# 1. Past Occupancy (T-1): What was the state in the previous interval?
df['prev_occupied_1'] = df.groupby('station_id')['occupied_slots'].shift(1)

# 2. Past Occupancy (T-2): Needed to calculate the trend of the past
df['prev_occupied_2'] = df.groupby('station_id')['occupied_slots'].shift(2)

# 3. FIX: occ_change (Momentum of the past: T-1 vs T-2)
# Tells the model: "Was it getting busier right before this moment?"
df['occ_change'] = df['prev_occupied_1'] - df['prev_occupied_2']

# 4. FIX: rolling_avg_3 (Stability of the past: Average of last 3 intervals before T)
df['rolling_avg_3'] = df.groupby('station_id')['occupied_slots'].shift(1).rolling(window=3).mean()

# Drop rows with NaNs (created by the shifts)
df_final = df.dropna()


In [4]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score

# 1. Select Features and Target
# We want to predict 'occupied_slots'
features = ['hour', 'day_of_week', 'is_weekend', 'total_slots', 
            'prev_occupied_1', 'occ_change', 'rolling_avg_3']

X = df_final[features]
y = df_final['occupied_slots']

# 2. Split (Using the last 20% of the 30 days as a test to see if it predicts 'future' well)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 3. Initialize and Train XGBoost
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    objective='reg:squarederror'
)

model.fit(X_train, y_train)

# 4. Accuracy Check
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"Model Evaluation Summary:")
print(f"1. Mean Absolute Error(MAE) : {mae:.4f}")
print(f"2. Root Mean Squared Error(RMSE) : {rmse:.4f}")
print(f"3. R² Score : {r2:.4f}")

Model Evaluation Summary:
1. Mean Absolute Error(MAE) : 0.3917
2. Root Mean Squared Error(RMSE) : 0.4446
3. R² Score : 0.2363


In [9]:
import pickle

# Save the trained XGBoost model to a file
with open('ev_xgboost_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model saved as ev_xgboost_model.pkl")


Model saved as ev_xgboost_model.pkl
